# 01 — Data collection

Pulls top positions for a list of Polymarket events, enriches each unique wallet with its lifetime leaderboard stats, and saves the result as a timestamped CSV in `data/raw/`.

**Input:** `events_to_scrape.txt` (one slug per line, `#` comments allowed).

**Output:** `data/raw/positions_<YYYYMMDD_HHMM>.csv` plus a `data/raw/wallets_<...>.csv` keyed by wallet.

If `events_to_scrape.txt` is empty, the notebook offers to auto-seed it with the top-N resolved events by volume from the Gamma API.

In [ ]:
from __future__ import annotations

import datetime as dt
import logging
from pathlib import Path

import pandas as pd

from polymarket_insider import api, scraper

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EVENTS_FILE = PROJECT_ROOT / "events_to_scrape.txt"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

TOP_N_PER_SIDE = 50           # how many holders per YES/NO side
AUTOSEED_IF_EMPTY = True      # if events_to_scrape.txt has no slugs, pull from API
AUTOSEED_N = 40               # how many events to auto-seed with

PROJECT_ROOT, RAW_DIR

## 1. Load slugs from `events_to_scrape.txt`

In [ ]:
def load_slugs(path: Path) -> list[str]:
    if not path.exists():
        return []
    out = []
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        out.append(line)
    return out

slugs = load_slugs(EVENTS_FILE)
print(f"Loaded {len(slugs)} slugs from {EVENTS_FILE.name}")
slugs[:10]

## 2. (optional) Auto-seed slugs from Gamma `/events`

If the file is empty we pull the top-N resolved events by volume and append them. You can re-run this cell selectively to refresh the seed list; existing slugs are preserved.

In [ ]:
if AUTOSEED_IF_EMPTY and not slugs:
    print(f"events_to_scrape.txt is empty - auto-seeding with top {AUTOSEED_N} resolved events by volume...")
    discovered = api.list_events(closed=True, archived=False, limit=AUTOSEED_N, order="volume")
    seeded = []
    for ev in discovered:
        slug = ev.get("slug")
        title = ev.get("title") or ev.get("question") or "?"
        if slug:
            seeded.append((slug, title, ev.get("volume")))

    # Append to the events file (preserving any user comments)
    existing_text = EVENTS_FILE.read_text(encoding="utf-8") if EVENTS_FILE.exists() else ""
    block = ["", "# --- auto-seeded (top resolved by volume) ---"]
    for slug, title, vol in seeded:
        block.append(f"{slug}  # {title} (vol={vol})")
    EVENTS_FILE.write_text(existing_text.rstrip() + "\n" + "\n".join(block) + "\n", encoding="utf-8")

    slugs = load_slugs(EVENTS_FILE)
    print(f"Now have {len(slugs)} slugs.")
else:
    print("Skipping auto-seed.")
slugs[:10]

## 3. Scrape positions for every slug

This is the expensive step. Expect roughly 1 second per event (one Gamma call + N market-positions calls). With the default rate-limit sleep of 0.4s, 40 events with 3 markets each ≈ 2-3 minutes.

In [ ]:
raw_df = scraper.scrape_events(slugs, top_n=TOP_N_PER_SIDE)
print(f"Scraped {len(raw_df)} position rows from {raw_df['event_slug'].nunique()} events.")
raw_df.head()

## 4. Enrich unique wallets with leaderboard totals

One call per unique wallet — cached locally so re-running is cheap.

In [ ]:
enriched_df = scraper.enrich_with_account_totals(raw_df, verbose=False)
print(f"Enriched {enriched_df['proxyWallet'].nunique()} unique wallets.")
enriched_df.head()

## 5. Save

Both the position-level table and a unique-wallets view land in `data/raw/`.

In [ ]:
stamp = dt.datetime.now().strftime("%Y%m%d_%H%M")
positions_path = RAW_DIR / f"positions_{stamp}.csv"
wallets_path = RAW_DIR / f"wallets_{stamp}.csv"

enriched_df.to_csv(positions_path, index=False, encoding="utf-8")

wallet_cols = [
    "proxyWallet",
    "name",
    "verified",
    "account_total_traded_volume",
    "account_total_gain_loss",
]
wallets_df = (
    enriched_df[wallet_cols]
    .drop_duplicates(subset=["proxyWallet"])
    .sort_values("account_total_traded_volume", ascending=False)
    .reset_index(drop=True)
)
wallets_df.to_csv(wallets_path, index=False, encoding="utf-8")

print(f"Saved {positions_path.relative_to(PROJECT_ROOT)}  ({len(enriched_df):,} rows)")
print(f"Saved {wallets_path.relative_to(PROJECT_ROOT)}  ({len(wallets_df):,} rows)")

## 6. Quick sanity check

In [ ]:
summary = pd.DataFrame(
    {
        "metric": [
            "events scraped",
            "markets covered",
            "position rows",
            "unique wallets",
            "verified wallets",
            "YES rows",
            "NO rows",
            "median totalBought ($)",
            "max totalBought ($)",
            "% positions with realizedPnl > 0",
        ],
        "value": [
            enriched_df["event_slug"].nunique(),
            enriched_df["condition_id"].nunique(),
            len(enriched_df),
            enriched_df["proxyWallet"].nunique(),
            int(enriched_df["verified"].fillna(False).astype(bool).sum()),
            int((enriched_df["side"] == "YES").sum()),
            int((enriched_df["side"] == "NO").sum()),
            round(float(enriched_df["totalBought"].median()), 2),
            round(float(enriched_df["totalBought"].max()), 2),
            round(float((enriched_df["realizedPnl"] > 0).mean() * 100), 1),
        ],
    }
)
summary